
# T — Submission robustness, mechanism generalization, family inference, and process-proximal validation

This notebook is a **new standalone analysis** for the final pre-submission checks.

It addresses four reviewer-facing questions in sequence:

1. **Interpolation robustness of the MgAl–PEI correction \(C\)**  
   Does the sign of the Beijing–Shenyang correction survive plausible temperature-interpolation choices between measured endpoint isotherms?

2. **Generalization of \(C\) beyond the Mg case**  
   Across class-C (nonlinear, rank-distinguishable) material × branch × temporal-mode cells, can preservation and reversal be described by
   \[
   \Delta H_{\rm full}=\Delta H_{\rm single}+C?
   \]

3. **Family-level uncertainty**  
   Re-estimate panel summaries after treating PEI-30/40/50 as one chemistry family rather than three independent materials.

4. **Process-proximal validation of \(H\)**  
   Compare equilibrium \(H\)-based climate ordering against a deliberately simple LDF dynamic-turnover surrogate. This is **not** a full process model; it tests whether the ordinal climate signal survives finite uptake kinetics.

The notebook preserves the existing Q/S2 support conventions wherever a comparison is made to earlier results.


In [ ]:

# Colab setup and paths
from google.colab import drive, files as colab_files
drive.mount('/content/drive')

from pathlib import Path
from itertools import combinations
import warnings, math, json, time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.interpolate import LinearNDInterpolator, interp1d
from scipy.spatial import Delaunay
from scipy.stats import kendalltau, spearmanr

EPS = 1e-12
RNG = np.random.default_rng(20261013)

DRIVE = Path('/content/drive/MyDrive')
preferred = DRIVE/'climate_screening_v1_3_1_uncertainty_audit'/'climate_screening_backbone'
if preferred.exists():
    PROJECT = preferred
else:
    hits = list(DRIVE.rglob('climate_screening_backbone'))
    if not hits:
        raise FileNotFoundError('climate_screening_backbone not found on Drive.')
    PROJECT = hits[0]

QDIR = PROJECT/'results/Q_confirmatory_nested_uncertainty_timescale'
S2DIR = PROJECT/'results/S2_trajectory_null_gate_standalone'
OUT = PROJECT/'results/T_submission_robustness_process_link'
FIG = PROJECT/'figures/T_submission_robustness_process_link'
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

CLIMATE = PROJECT/'data/processed/climate_hourly_2015_2024.csv'
PEI = PROJECT/'data/raw/PEI_climate_measured_capacity.csv'

def first_existing(patterns, roots=(DRIVE,)):
    for root in roots:
        for pat in patterns:
            hits = list(root.rglob(pat))
            if hits:
                return hits[0]
    return None

MASTER = first_existing([
    'q_RH_T_master_cycle08_audited_20261010(1).xlsx',
    'q_RH_T_master_cycle08_audited_20261010.xlsx',
    'q_RH_T_master_combined_v2(1).xlsx',
    'q_RH_T_master_combined_v2.xlsx',
])

if MASTER is None:
    print('Upload the equilibrium master workbook used for Q.')
    up = colab_files.upload()
    xls = [n for n in up if n.lower().endswith(('.xlsx','.xls'))]
    if not xls:
        raise FileNotFoundError('Equilibrium master workbook is required.')
    MASTER = Path('/content')/xls[0]

def resolve_or_upload(candidates, label):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    print(f'Upload {label}')
    up = colab_files.upload()
    for n in up:
        if any(Path(n).name == Path(c).name for c in candidates):
            return Path('/content')/n
    if len(up) == 1:
        return Path('/content')/next(iter(up))
    raise FileNotFoundError(label)

NESTED_INCREMENTAL = resolve_or_upload(
    [QDIR/'nested_incremental_value.csv'],
    'nested_incremental_value.csv'
)

S2_GATE = resolve_or_upload(
    [
        S2DIR/'S2_gate_by_material.csv',
        PROJECT/'results/S2_trajectory_null_gate_standalone'/'S2_gate_by_material.csv',
    ],
    'S2_gate_by_material.csv'
)

S2_EQUIV = resolve_or_upload(
    [
        S2DIR/'S2_null_equivalence_diagnostic_all'/'S2_null_equivalence_all_groups.csv',
        PROJECT/'results/S2_trajectory_null_gate_standalone'/'S2_null_equivalence_all_groups.csv',
    ],
    'S2_null_equivalence_all_groups.csv'
)

print('PROJECT:', PROJECT)
print('MASTER :', MASTER)
print('OUT    :', OUT)


## 1. Rebuild the equilibrium representations and exact Q support

In [ ]:

def normalize_columns(df):
    d = df.copy()
    ren = {}
    if 'T_C' in d.columns: ren['T_C'] = 'temperature_C'
    if 'RH_pct' in d.columns: ren['RH_pct'] = 'rh_pct'
    if 'RH' in d.columns and 'rh_pct' not in d.columns: ren['RH'] = 'rh_pct'
    if 'q' in d.columns and 'q_mmol_g' not in d.columns: ren['q'] = 'q_mmol_g'
    if 'q_mmol_g_mean' in d.columns and 'q_mmol_g' not in d.columns:
        ren['q_mmol_g_mean'] = 'q_mmol_g'
    d = d.rename(columns=ren)
    if 'temperature_C' not in d.columns and 'temperature_K' in d.columns:
        d['temperature_C'] = d.temperature_K - 273.15
    return d

UNCERTAINTY_CANDIDATES = [
    'q_sd','q_std','q_sigma','q_uncertainty',
    'uncertainty_mmol_g','sd_mmol_g','std_mmol_g',
    'q_error','q_sem'
]

def clean(df):
    d = normalize_columns(df)
    need = ['temperature_C','rh_pct','q_mmol_g']
    if not set(need).issubset(d.columns):
        return pd.DataFrame(columns=need+['q_sigma'])
    sigcol = next((c for c in UNCERTAINTY_CANDIDATES if c in d.columns), None)
    cols = need + ([sigcol] if sigcol else [])
    d = d[cols].replace([np.inf,-np.inf],np.nan).dropna(subset=need)
    agg = {'q_mmol_g':'mean'}
    if sigcol: agg[sigcol] = 'mean'
    d = d.groupby(['temperature_C','rh_pct'],as_index=False).agg(agg)
    d = d.rename(columns={sigcol:'q_sigma'}) if sigcol else d.assign(q_sigma=np.nan)
    return d

climate = pd.read_csv(
    CLIMATE,
    usecols=['climate_class','city','timestamp_utc','temperature_C','RH_pct']
)
climate['timestamp_utc'] = pd.to_datetime(climate.timestamp_utc, utc=True, errors='coerce')
climate = climate.dropna().sort_values(
    ['climate_class','city','timestamp_utc']
).reset_index(drop=True)

materials = {}

if PEI.exists():
    d = pd.read_csv(PEI)
    if 'material' in d.columns:
        for m,g in d.groupby('material'):
            z = clean(g)
            if len(z):
                materials[str(m)] = z

raw = pd.read_excel(MASTER, sheet_name='Raw_Data')
mcol = 'Material' if 'Material' in raw.columns else 'material'
for m,g in raw.groupby(mcol):
    z = clean(g)
    if len(z) and (str(m) not in materials or len(z) > len(materials[str(m)])):
        materials[str(m)] = z

for pth in [
    PROJECT/'data/processed/validation_IRA_RHT.csv',
    PROJECT/'data/raw/validation_IRA_RHT.csv',
    PROJECT/'data/raw/IRA_A500_RHT.csv',
]:
    if pth.exists():
        d = pd.read_csv(pth)
        if 'material' in d.columns:
            for m,g in d.groupby('material'):
                z = clean(g)
                if len(z) and (str(m) not in materials or len(z) > len(materials[str(m)])):
                    materials[str(m)] = z

rows = []
for m,d in materials.items():
    temps = sorted(d.temperature_C.unique())
    if len(temps) < 3:
        continue
    Tlow, Thigh = temps[0], temps[-1]
    Tref = min(temps, key=lambda x: abs(x - 0.5*(Tlow+Thigh)))
    if Tref in [Tlow, Thigh]:
        continue
    nlow = len(d[d.temperature_C == Tlow])
    nref = len(d[d.temperature_C == Tref])
    nhigh = len(d[d.temperature_C == Thigh])
    if min(nlow,nref,nhigh) < 2:
        continue
    rows.append(dict(
        material=m, Tlow=Tlow, Tref=Tref, Thigh=Thigh,
        n_low=nlow, n_ref=nref, n_high=nhigh
    ))

elig = pd.DataFrame(rows)
display(elig)


In [ ]:

class FullSurface:
    def __init__(self,d):
        pts = d[['rh_pct','temperature_C']].to_numpy(float)
        vals = d.q_mmol_g.to_numpy(float)
        self.tri = Delaunay(pts)
        self.fun = LinearNDInterpolator(self.tri, vals, fill_value=np.nan)
    def eval(self,rh,T):
        return np.asarray(self.fun(np.asarray(rh,float), np.asarray(T,float)), float)

class SingleIso:
    def __init__(self,d,Tref):
        z = d[d.temperature_C == Tref].sort_values('rh_pct')
        self.Tref = float(Tref)
        self.rmin = float(z.rh_pct.min())
        self.rmax = float(z.rh_pct.max())
        self.fun = interp1d(
            z.rh_pct,z.q_mmol_g,kind='linear',
            bounds_error=False,fill_value=np.nan
        )
    def eval(self,rh,T):
        return np.asarray(self.fun(np.asarray(rh,float)),float)

class PairSlice:
    def __init__(self,d,T1,T2):
        self.T1 = float(min(T1,T2))
        self.T2 = float(max(T1,T2))
        z1 = d[d.temperature_C == self.T1].sort_values('rh_pct')
        z2 = d[d.temperature_C == self.T2].sort_values('rh_pct')
        self.rmin = max(float(z1.rh_pct.min()), float(z2.rh_pct.min()))
        self.rmax = min(float(z1.rh_pct.max()), float(z2.rh_pct.max()))
        self.f1 = interp1d(z1.rh_pct,z1.q_mmol_g,kind='linear',bounds_error=False,fill_value=np.nan)
        self.f2 = interp1d(z2.rh_pct,z2.q_mmol_g,kind='linear',bounds_error=False,fill_value=np.nan)
    def eval(self,rh,T):
        rh = np.asarray(rh,float)
        T = np.asarray(T,float)
        q1 = np.asarray(self.f1(rh),float)
        q2 = np.asarray(self.f2(rh),float)
        lam = (T-self.T1)/(self.T2-self.T1)
        out = (1-lam)*q1 + lam*q2
        out[(T<self.T1)|(T>self.T2)] = np.nan
        return out

class ThreeSlice:
    def __init__(self,d,Tlow,Tref,Thigh):
        self.Tlow, self.Tref, self.Thigh = map(float,(Tlow,Tref,Thigh))
        zl = d[d.temperature_C == Tlow].sort_values('rh_pct')
        zr = d[d.temperature_C == Tref].sort_values('rh_pct')
        zh = d[d.temperature_C == Thigh].sort_values('rh_pct')
        self.rmin = max(float(zl.rh_pct.min()),float(zr.rh_pct.min()),float(zh.rh_pct.min()))
        self.rmax = min(float(zl.rh_pct.max()),float(zr.rh_pct.max()),float(zh.rh_pct.max()))
        self.fl = interp1d(zl.rh_pct,zl.q_mmol_g,bounds_error=False,fill_value=np.nan)
        self.fr = interp1d(zr.rh_pct,zr.q_mmol_g,bounds_error=False,fill_value=np.nan)
        self.fh = interp1d(zh.rh_pct,zh.q_mmol_g,bounds_error=False,fill_value=np.nan)
    def eval(self,rh,T):
        rh = np.asarray(rh,float); T = np.asarray(T,float)
        ql = np.asarray(self.fl(rh),float)
        qr = np.asarray(self.fr(rh),float)
        qh = np.asarray(self.fh(rh),float)
        out = np.full(np.broadcast(rh,T).shape,np.nan,float)
        lo = (T>=self.Tlow)&(T<=self.Tref)
        hi = (T>=self.Tref)&(T<=self.Thigh)
        if np.any(lo):
            lam = (T[lo]-self.Tlow)/(self.Tref-self.Tlow)
            out[lo] = (1-lam)*ql[lo] + lam*qr[lo]
        if np.any(hi):
            lam = (T[hi]-self.Tref)/(self.Thigh-self.Tref)
            out[hi] = (1-lam)*qr[hi] + lam*qh[hi]
        return out

model = {}
for _,r in elig.iterrows():
    d = materials[r.material]
    model[r.material] = dict(
        full=FullSurface(d),
        single_ref=SingleIso(d,r.Tref),
        nested_low=PairSlice(d,r.Tlow,r.Tref),
        nested_high=PairSlice(d,r.Tref,r.Thigh),
        three_slice=ThreeSlice(d,r.Tlow,r.Tref,r.Thigh),
    )


In [ ]:

def subsample_climate(df, step, offset=0):
    out = []
    for (cc,city),g in df.groupby(['climate_class','city'], sort=False):
        z = g.sort_values('timestamp_utc').iloc[offset::step].copy().reset_index(drop=True)
        if len(z):
            z['climate_class'] = cc
            z['city'] = city
            out.append(z)
    return pd.concat(out,ignore_index=True) if out else pd.DataFrame()

def average_climate(df, hours, offset=0):
    out = []
    shift = pd.Timedelta(hours=int(offset))
    for (cc,city),g in df.groupby(['climate_class','city'],sort=False):
        z = g[['timestamp_utc','temperature_C','RH_pct']].copy()
        z['timestamp_shifted'] = z.timestamp_utc - shift
        z = (
            z.set_index('timestamp_shifted')[['temperature_C','RH_pct']]
             .resample(f'{hours}h').mean().dropna().reset_index()
             .rename(columns={'timestamp_shifted':'timestamp_utc'})
        )
        z['timestamp_utc'] = z.timestamp_utc + shift
        z['climate_class'] = cc
        z['city'] = city
        out.append(z)
    return pd.concat(out,ignore_index=True) if out else pd.DataFrame()

BASE_MODES = {
    'subsample_1h': subsample_climate(climate,1,0),
    'subsample_3h': subsample_climate(climate,3,0),
    'subsample_6h': subsample_climate(climate,6,0),
    'subsample_24h': subsample_climate(climate,24,0),
    'average_6h': average_climate(climate,6,0),
    'average_24h': average_climate(climate,24,0),
}

def base_transitions(cdf,Tlo,Thi,Rlo,Rhi):
    out = []
    for (cc,city),g in cdf.groupby(['climate_class','city'],sort=False):
        g = g.sort_values('timestamp_utc').reset_index(drop=True)
        rh = g.RH_pct.to_numpy(float)
        T = g.temperature_C.to_numpy(float)
        ts = g.timestamp_utc.to_numpy()
        ok = (
            (rh[:-1]>=Rlo)&(rh[:-1]<=Rhi)&
            (rh[1:]>=Rlo)&(rh[1:]<=Rhi)&
            (T[:-1]>=Tlo)&(T[:-1]<=Thi)&
            (T[1:]>=Tlo)&(T[1:]<=Thi)
        )
        ix = np.where(ok)[0]
        if len(ix):
            out.append(pd.DataFrame({
                'transition_id':[f'{cc}|{city}|{pd.Timestamp(ts[k]).isoformat()}' for k in ix],
                'climate_class':cc,'city':city,'seq_idx':ix,
                'RH0':rh[ix],'RH1':rh[ix+1],'T0':T[ix],'T1':T[ix+1],
                'timestamp_utc':pd.to_datetime(ts[ix],utc=True),
            }))
    return pd.concat(out,ignore_index=True) if out else pd.DataFrame()

def exact_q_common_mask(base,md,branch):
    reps = (
        ['full','single_ref','nested_low','three_slice']
        if branch=='lower'
        else ['full','single_ref','nested_high','three_slice']
    )
    masks = []
    for rep in reps:
        obj = md[rep]
        q0 = obj.eval(base.RH0.to_numpy(float),base.T0.to_numpy(float))
        q1 = obj.eval(base.RH1.to_numpy(float),base.T1.to_numpy(float))
        masks.append(np.isfinite(q0)&np.isfinite(q1))
    return np.logical_and.reduce(masks)

def exact_q_R_support(md,branch):
    reps = (
        ['full','single_ref','nested_low','three_slice']
        if branch=='lower'
        else ['full','single_ref','nested_high','three_slice']
    )
    rmins,rmaxs = [],[]
    for rep in reps:
        obj = md[rep]
        if hasattr(obj,'rmin'):
            rmins.append(float(obj.rmin)); rmaxs.append(float(obj.rmax))
    return max([0.0]+rmins), min([100.0]+rmaxs)

def branch_bounds(er,branch):
    if branch=='lower':
        return float(er.Tlow),float(er.Tref)
    if branch=='upper':
        return float(er.Tref),float(er.Thigh)
    raise ValueError(branch)

def score_H_from_model(z,obj):
    q0 = obj.eval(z.RH0.to_numpy(float),z.T0.to_numpy(float))
    q1 = obj.eval(z.RH1.to_numpy(float),z.T1.to_numpy(float))
    ok = np.isfinite(q0)&np.isfinite(q1)
    x = z.loc[ok,['climate_class','city','transition_id','seq_idx']].copy()
    x['dq'] = q1[ok]-q0[ok]
    return (
        x.groupby(['climate_class','city'],as_index=False)
         .agg(N=('transition_id','size'),H=('dq',lambda v:0.5*np.abs(v).sum()))
    )

def exact_problem(material,branch,mode_df):
    er = elig.set_index('material').loc[material]
    Tlo,Thi = branch_bounds(er,branch)
    md = model[material]
    Rlo,Rhi = exact_q_R_support(md,branch)
    base = base_transitions(mode_df,Tlo,Thi,Rlo,Rhi)
    if base.empty:
        return base
    return base.loc[exact_q_common_mask(base,md,branch)].copy()

print('Core reconstruction ready.')



## 2. Test A — MgAl–PEI \(C\) interpolation sensitivity

The measured endpoints are fixed. Only the temperature-direction interpolation between the two measured lower-branch isotherms is changed.

The primary quantity is
\[
C=(H_B^{\rm alt}-H_S^{\rm alt})-(H_B^{\rm single}-H_S^{\rm single}).
\]

The test is **sign robustness**, not equality of intermediate-temperature magnitudes.

Schemes:
- `Q_LinearND`: original fuller interpolated Q reference
- `endpoint_linear`: linear temperature weight between measured endpoint isotherms
- `endpoint_sqrt`: concave temperature weight
- `endpoint_square`: convex temperature weight
- `endpoint_smoothstep`: smooth monotone endpoint-preserving weight


In [ ]:

class EndpointWarpSurface:
    def __init__(self,d,T1,T2,warp='linear'):
        self.T1 = float(min(T1,T2)); self.T2 = float(max(T1,T2))
        z1 = d[d.temperature_C==self.T1].sort_values('rh_pct')
        z2 = d[d.temperature_C==self.T2].sort_values('rh_pct')
        self.rmin = max(float(z1.rh_pct.min()),float(z2.rh_pct.min()))
        self.rmax = min(float(z1.rh_pct.max()),float(z2.rh_pct.max()))
        self.f1 = interp1d(z1.rh_pct,z1.q_mmol_g,kind='linear',bounds_error=False,fill_value=np.nan)
        self.f2 = interp1d(z2.rh_pct,z2.q_mmol_g,kind='linear',bounds_error=False,fill_value=np.nan)
        self.warp = warp
    def weight(self,lam):
        lam = np.asarray(lam,float)
        if self.warp=='linear':
            return lam
        if self.warp=='sqrt':
            return np.sqrt(np.clip(lam,0,1))
        if self.warp=='square':
            return np.clip(lam,0,1)**2
        if self.warp=='smoothstep':
            x = np.clip(lam,0,1)
            return 3*x*x - 2*x*x*x
        raise ValueError(self.warp)
    def eval(self,rh,T):
        rh=np.asarray(rh,float); T=np.asarray(T,float)
        q1=np.asarray(self.f1(rh),float); q2=np.asarray(self.f2(rh),float)
        lam=(T-self.T1)/(self.T2-self.T1)
        w=self.weight(lam)
        out=(1-w)*q1+w*q2
        out[(T<self.T1)|(T>self.T2)] = np.nan
        return out

MG = 'Mg0.55Al-CO3_PEI(30)'
if MG not in materials:
    raise KeyError(f'{MG} not found in loaded materials.')

er = elig.set_index('material').loc[MG]
mg_schemes = {
    'Q_LinearND': model[MG]['full'],
    'endpoint_linear': EndpointWarpSurface(materials[MG],er.Tlow,er.Tref,'linear'),
    'endpoint_sqrt': EndpointWarpSurface(materials[MG],er.Tlow,er.Tref,'sqrt'),
    'endpoint_square': EndpointWarpSurface(materials[MG],er.Tlow,er.Tref,'square'),
    'endpoint_smoothstep': EndpointWarpSurface(materials[MG],er.Tlow,er.Tref,'smoothstep'),
}


In [ ]:

A_CC,A_CITY = 'monsoonal_continental','beijing'
B_CC,B_CITY = 'monsoonal_continental','shenyang'

phase_specs = []
for off in range(3):
    phase_specs.append(('subsample_3h',off,subsample_climate(climate,3,off)))
for off in range(6):
    phase_specs.append(('subsample_6h',off,subsample_climate(climate,6,off)))
for off in range(6):
    phase_specs.append(('average_6h',off,average_climate(climate,6,off)))

def pair_delta(score_df,ccA=A_CC,cityA=A_CITY,ccB=B_CC,cityB=B_CITY):
    a = score_df[(score_df.climate_class==ccA)&(score_df.city==cityA)]
    b = score_df[(score_df.climate_class==ccB)&(score_df.city==cityB)]
    if len(a)!=1 or len(b)!=1:
        return np.nan
    return float(a.H.iloc[0]-b.H.iloc[0])

mg_rows=[]
mg_cell_rows=[]

for family,off,cdf in phase_specs:
    z = exact_problem(MG,'lower',cdf)
    if z.empty:
        continue

    single = score_H_from_model(z,model[MG]['single_ref'])
    d_single = pair_delta(single)

    z2=z.copy()
    z2['T_mid']=0.5*(z2.T0+z2.T1)
    z2['RH_mid']=0.5*(z2.RH0+z2.RH1)

    for scheme,obj in mg_schemes.items():
        alt = score_H_from_model(z,obj)
        d_alt = pair_delta(alt)
        C = d_alt-d_single

        # additive transition correction, used for the low-T/high-RH concentration test
        qa0=obj.eval(z2.RH0.to_numpy(float),z2.T0.to_numpy(float))
        qa1=obj.eval(z2.RH1.to_numpy(float),z2.T1.to_numpy(float))
        qs0=model[MG]['single_ref'].eval(z2.RH0.to_numpy(float),z2.T0.to_numpy(float))
        qs1=model[MG]['single_ref'].eval(z2.RH1.to_numpy(float),z2.T1.to_numpy(float))
        corr=0.5*(np.abs(qa1-qa0)-np.abs(qs1-qs0))
        zz=z2[['climate_class','city','T_mid','RH_mid']].copy()
        zz['corr']=corr
        region=(zz.T_mid < -10)&(zz.RH_mid>=40)&(zz.RH_mid<=70)

        def city_sum(mask,cc,city):
            return float(zz.loc[mask&(zz.climate_class==cc)&(zz.city==city),'corr'].sum())

        C_region = (
            city_sum(region,A_CC,A_CITY) -
            city_sum(region,B_CC,B_CITY)
        )

        mg_rows.append(dict(
            phase_family=family,offset_h=off,scheme=scheme,
            delta_Hsingle=d_single,delta_Halt=d_alt,C=C,
            sign_C=np.sign(C),
            region_C=C_region,
            region_fraction_signed=(C_region/C if abs(C)>EPS else np.nan),
            reversal=(np.sign(d_single)!=np.sign(d_alt)),
        ))

mg_interp = pd.DataFrame(mg_rows)
mg_interp.to_csv(OUT/'Mg_C_interpolation_sensitivity_by_phase.csv',index=False)

mg_interp_summary = (
    mg_interp.groupby('scheme',as_index=False)
    .agg(
        n_phases=('C','size'),
        n_C_negative=('C',lambda x:int(np.sum(np.asarray(x)<0))),
        frac_C_negative=('C',lambda x:float(np.mean(np.asarray(x)<0))),
        C_min=('C','min'),C_median=('C','median'),C_max=('C','max'),
        region_fraction_median=('region_fraction_signed','median'),
        reversal_fraction=('reversal','mean'),
    )
)
mg_interp_summary.to_csv(OUT/'Mg_C_interpolation_sensitivity_summary.csv',index=False)
display(mg_interp_summary)

fig,ax=plt.subplots(figsize=(8,4.8))
for scheme,g in mg_interp.groupby('scheme'):
    x=np.arange(len(g))
    ax.plot(x,g.C.to_numpy(float),marker='o',label=scheme)
ax.axhline(0,linewidth=1)
ax.set_ylabel('C = ΔH_alt - ΔH_single')
ax.set_xlabel('Temporal phase (ordered within scheme)')
ax.set_title('Mg lower: sign robustness of pair correction C')
ax.legend(fontsize=8,ncol=2)
fig.tight_layout()
fig.savefig(FIG/'Mg_C_interpolation_sensitivity.png',dpi=220)
plt.show()

if (mg_interp_summary.frac_C_negative==1).all():
    print('PASS: C is negative for every tested phase under every interpolation scheme.')
else:
    print('CAUTION: at least one interpolation scheme changes the sign of C.')



### Interpretation rule for Test A

- **Strong mechanism claim:** all plausible endpoint-preserving interpolation schemes retain the same sign of \(C\); only magnitude changes.
- **Moderate claim:** most schemes retain the sign, but one plausible scheme crosses zero.
- **Weak claim:** sign depends materially on interpolation choice.

The statement should remain endpoint-based: the measured low-temperature and reference isotherms differ; the intermediate-temperature magnitude is model-dependent.


## 3. Test B — Generalize the \(C\) decomposition across class-C cells

In [ ]:

diag = pd.read_csv(S2_EQUIV)
gate = pd.read_csv(S2_GATE)

# tolerate either naming used by the two S2 implementations
if 'delta_rho_partial' not in gate.columns and 'delta_rho_partial_single_minus_null' in gate.columns:
    gate['delta_rho_partial'] = gate['delta_rho_partial_single_minus_null']
if 'strict_CI_pass' not in gate.columns and 'strict_gate_pass' in gate.columns:
    gate['strict_CI_pass'] = gate['strict_gate_pass']

Ccells = diag[diag.mechanism_class.astype(str).str.startswith('C_')].copy()
print('Class-C material × branch × mode cells:', len(Ccells))
display(Ccells[['material','branch','temporal_mode']].head())


In [ ]:

def score_pair_decomposition(material,branch,mode):
    if mode not in BASE_MODES:
        return pd.DataFrame(),pd.DataFrame()
    z = exact_problem(material,branch,BASE_MODES[mode])
    if z.empty:
        return pd.DataFrame(),pd.DataFrame()

    sf=score_H_from_model(z,model[material]['full']).rename(columns={'H':'H_full','N':'N_full'})
    ss=score_H_from_model(z,model[material]['single_ref']).rename(columns={'H':'H_single','N':'N_single'})
    g=sf.merge(ss,on=['climate_class','city'],how='inner')
    g=g[g.N_full==g.N_single].copy()
    g['E_climate']=g.H_full-g.H_single

    pair=[]
    for i,j in combinations(range(len(g)),2):
        a=g.iloc[i]; b=g.iloc[j]
        ds=float(a.H_single-b.H_single)
        df=float(a.H_full-b.H_full)
        C=float((a.E_climate-b.E_climate))
        margin=abs(ds)/(0.5*(abs(a.H_single)+abs(b.H_single))+EPS)
        reversal=(np.sign(ds)!=0 and np.sign(df)!=0 and np.sign(ds)!=np.sign(df))
        criterion=(
            np.sign(C)!=0 and np.sign(ds)!=0 and
            np.sign(C)!=np.sign(ds) and abs(C)>abs(ds)
        )
        pair.append(dict(
            material=material,branch=branch,temporal_mode=mode,
            climate_A=f'{a.climate_class}||{a.city}',
            climate_B=f'{b.climate_class}||{b.city}',
            delta_Hsingle=ds,delta_Hfull=df,C=C,
            absC_over_absSingle=abs(C)/(abs(ds)+EPS),
            single_relative_margin=margin,
            reversal=reversal,
            criterion_predicts_reversal=criterion,
            preserved_same_sign=(not reversal),
        ))
    return g,pd.DataFrame(pair)

climate_tables=[]
pair_tables=[]

for _,r in Ccells.iterrows():
    m,b,mode=r.material,r.branch,r.temporal_mode
    if m not in model or mode not in BASE_MODES:
        continue
    gc,gp=score_pair_decomposition(m,b,mode)
    if len(gc):
        gc['material']=m; gc['branch']=b; gc['temporal_mode']=mode
        climate_tables.append(gc)
    if len(gp):
        pair_tables.append(gp)

C_climate = pd.concat(climate_tables,ignore_index=True) if climate_tables else pd.DataFrame()
C_pairs = pd.concat(pair_tables,ignore_index=True) if pair_tables else pd.DataFrame()

C_climate.to_csv(OUT/'C_generalization_climate_level.csv',index=False)
C_pairs.to_csv(OUT/'C_generalization_pair_level.csv',index=False)

if len(C_pairs):
    # exact logical identity audit, ignoring true zero ties
    audit=C_pairs[
        (np.abs(C_pairs.delta_Hsingle)>1e-12)&
        (np.abs(C_pairs.delta_Hfull)>1e-12)
    ].copy()
    audit['criterion_matches_observed']=audit.reversal==audit.criterion_predicts_reversal
    print('Exact reversal-condition agreement:',
          audit.criterion_matches_observed.mean(),
          f"({audit.criterion_matches_observed.sum()}/{len(audit)})")

    summary=(
        C_pairs.groupby(['material','branch','temporal_mode'],as_index=False)
        .agg(
            n_pairs=('reversal','size'),
            reversal_fraction=('reversal','mean'),
            median_absC_over_absSingle=('absC_over_absSingle','median'),
            q90_absC_over_absSingle=('absC_over_absSingle',lambda x:np.nanquantile(x,0.90)),
            median_single_margin=('single_relative_margin','median'),
            frac_C_opposes_single=(
                'C',
                lambda x: np.nan
            ),
        )
    )
    # compute opposition separately because it requires two columns
    opp=(
        C_pairs.assign(
            opposes=lambda d:
            (np.sign(d.C)!=0)&(np.sign(d.delta_Hsingle)!=0)&
            (np.sign(d.C)!=np.sign(d.delta_Hsingle))
        )
        .groupby(['material','branch','temporal_mode'],as_index=False)
        .opposes.mean()
        .rename(columns={'opposes':'frac_C_opposes_single'})
    )
    summary=summary.drop(columns='frac_C_opposes_single').merge(
        opp,on=['material','branch','temporal_mode'],how='left'
    )
    summary.to_csv(OUT/'C_generalization_summary.csv',index=False)
    display(summary.sort_values(['material','branch','temporal_mode']).head(50))


In [ ]:

# Preservation vs failure as a function of correction-to-margin ratio
if len(C_pairs):
    d=C_pairs[np.isfinite(C_pairs.absC_over_absSingle)].copy()
    bins=[0,0.25,0.5,1,2,5,np.inf]
    labels=['<0.25','0.25-0.5','0.5-1','1-2','2-5','>5']
    d['ratio_bin']=pd.cut(d.absC_over_absSingle,bins=bins,labels=labels,right=False)
    ratio_summary=(
        d.groupby('ratio_bin',observed=True,as_index=False)
         .agg(n_pairs=('reversal','size'),reversal_fraction=('reversal','mean'))
    )
    ratio_summary.to_csv(OUT/'C_ratio_vs_reversal.csv',index=False)
    display(ratio_summary)

    fig,ax=plt.subplots(figsize=(6.5,4))
    ax.bar(ratio_summary.ratio_bin.astype(str),ratio_summary.reversal_fraction)
    ax.set_ylim(0,1)
    ax.set_ylabel('Observed reversal fraction')
    ax.set_xlabel('|C| / |ΔH_single|')
    ax.set_title('Class-C cells: correction size versus pair reversal')
    fig.tight_layout()
    fig.savefig(FIG/'C_ratio_vs_reversal.png',dpi=220)
    plt.show()



**Important:** the reversal condition is an identity once the fuller surface is known. Its scientific value is explanatory, not predictive. A practical stopping rule would require a pre-measurement bound or prior for \(|C|\).


## 4. Test C — Family-level resampling

In [ ]:

def chemistry_family(m):
    m=str(m)
    if m in {'PEI-30wt','PEI-40wt','PEI-50wt'}:
        return 'PEI_loading_family'
    if 'IRA-900' in m:
        return 'IRA_900'
    if m.startswith('Mg0.55Al'):
        return 'MgAl_PEI'
    if 'Al2O3' in m:
        return 'Al2O3_TEPA'
    if 'MIL-101' in m:
        return 'MIL101_TEPA'
    if 'ImCOF' in m:
        return 'ImCOF_TAEA'
    if 'LWT' in m:
        return 'LWT_bPPI5'
    return m

def family_bootstrap(df,value_col,group_cols,n_boot=5000,seed=20261013):
    rng=np.random.default_rng(seed)
    out=[]
    for key,g in df.groupby(group_cols,dropna=False):
        if not isinstance(key,tuple): key=(key,)
        x=g[['family',value_col]].dropna().copy()
        fam=(
            x.groupby('family',as_index=False)[value_col]
             .median()
        )
        vals=fam[value_col].to_numpy(float)
        names=fam.family.tolist()
        if len(vals)==0:
            continue
        obs=float(np.median(vals))
        boots=np.empty(n_boot,float)
        for b in range(n_boot):
            ix=rng.integers(0,len(vals),len(vals))
            boots[b]=np.median(vals[ix])
        row={c:v for c,v in zip(group_cols,key)}
        row.update(
            n_families=len(vals),
            observed_family_median=obs,
            ci_lo=float(np.quantile(boots,.025)),
            ci_hi=float(np.quantile(boots,.975)),
            p_positive=float(np.mean(boots>0)),
            families='; '.join(names),
        )
        out.append(row)
    return pd.DataFrame(out)

inc=pd.read_csv(NESTED_INCREMENTAL)
inc['family']=inc.material.map(chemistry_family)

nested_family_boot=family_bootstrap(
    inc,
    'tau_gain',
    ['branch','temporal_mode','metric'],
    n_boot=5000
)
nested_family_boot.to_csv(OUT/'family_bootstrap_nested_tau_gain.csv',index=False)

# Class-C null-gate family inference
cg=gate.merge(
    Ccells[['material','branch','temporal_mode']].drop_duplicates(),
    on=['material','branch','temporal_mode'],
    how='inner'
)
cg['family']=cg.material.map(chemistry_family)

C_family_boot=family_bootstrap(
    cg,
    'delta_rho_partial',
    ['branch','temporal_mode'],
    n_boot=5000
)
C_family_boot.to_csv(OUT/'family_bootstrap_C_gate_delta_rho.csv',index=False)

print('Nested hierarchy: family-level tau-gain bootstrap')
display(nested_family_boot)

print('\nClass-C trajectory-null gate: family-level delta-rho bootstrap')
display(C_family_boot)

print('\nIndependent chemistry count in class C:',
      cg.family.nunique(), sorted(cg.family.unique()))



With only three independent class-C chemistry groups, these intervals are **descriptive sensitivity intervals**, not strong asymptotic inference. The main role is to show how much formulation-level precision was inflated by counting the three PEI loadings separately.



## 5. Test D — Does \(H\) preserve climate ordering under finite kinetics?

A reviewer can reasonably ask why an equilibrium total-variation score matters for a process.

We therefore introduce a deliberately minimal process-proximal comparator:

\[
\frac{dq}{dt}=k(q^*-q),\qquad \tau_k=1/k.
\]

For each contiguous supported weather segment, the dynamic loading state relaxes toward the fuller equilibrium \(q^*(RH,T)\). We then calculate

\[
H_{\rm LDF}=\frac12\sum_t |q_{t+1}-q_t|
\]

and a balanced directional turnover,

\[
H_{\rm LDF,bal}=\min\left(\sum \max(\Delta q,0),\sum \max(-\Delta q,0)\right).
\]

This is **not productivity and not a complete DAC cycle model**. It tests whether the climate ordering induced by equilibrium opportunity survives a simple kinetic bottleneck.


In [ ]:

PROCESS_MATERIALS = [
    m for m in ['IRA-900-C','Mg0.55Al-CO3_PEI(30)']
    if m in model
]
PROCESS_BRANCHES = ['lower','upper']
TAU_HOURS = [0.25,0.5,1,3,6,12]
PROCESS_MODE = 'subsample_1h'

def contiguous_segments(z):
    z=z.sort_values(['climate_class','city','seq_idx']).copy()
    segs=[]
    for (cc,city),g in z.groupby(['climate_class','city'],sort=False):
        g=g.sort_values('seq_idx')
        seq=g.seq_idx.to_numpy(int)
        cuts=np.r_[0,np.where(np.diff(seq)!=1)[0]+1,len(g)]
        for a,b in zip(cuts[:-1],cuts[1:]):
            if b>a:
                segs.append((cc,city,g.iloc[a:b].copy()))
    return segs

def ldf_scores_for_problem(material,branch,cdf,tau_h):
    z=exact_problem(material,branch,cdf)
    if z.empty:
        return pd.DataFrame()

    obj=model[material]['full']
    rows=[]
    by_climate={}

    for cc,city,g in contiguous_segments(z):
        rh0=g.RH0.to_numpy(float); rh1=g.RH1.to_numpy(float)
        T0=g.T0.to_numpy(float); T1=g.T1.to_numpy(float)
        q0s=obj.eval(rh0,T0); q1s=obj.eval(rh1,T1)
        ok=np.isfinite(q0s)&np.isfinite(q1s)
        if not ok.all():
            g=g.loc[ok].copy()
            q0s=q0s[ok];q1s=q1s[ok]
        if len(g)==0:
            continue

        # equilibrium H on exactly the same transitions
        H_eq=0.5*np.abs(q1s-q0s).sum()

        # dt from sampling timestamps; fallback to 1 h if unavailable
        ts=pd.to_datetime(g.timestamp_utc,utc=True)
        if len(ts)>=2:
            dts=np.diff(ts.view('int64'))/3.6e12
            dt_default=float(np.nanmedian(dts)) if len(dts) else 1.0
        else:
            dt_default=1.0

        q=float(q0s[0])
        pos=neg=tv=0.0
        for qi_star in q1s:
            alpha=1.0-np.exp(-dt_default/max(float(tau_h),1e-9))
            qn=q + alpha*(float(qi_star)-q)
            dq=qn-q
            tv += abs(dq)
            pos += max(dq,0.0)
            neg += max(-dq,0.0)
            q=qn

        key=(cc,city)
        if key not in by_climate:
            by_climate[key]=dict(H_eq=0.0,H_LDF=0.0,H_LDF_bal=0.0,N=0)
        by_climate[key]['H_eq'] += H_eq
        by_climate[key]['H_LDF'] += 0.5*tv
        by_climate[key]['H_LDF_bal'] += min(pos,neg)
        by_climate[key]['N'] += len(g)

    for (cc,city),v in by_climate.items():
        rows.append(dict(climate_class=cc,city=city,**v))
    return pd.DataFrame(rows)

process_rows=[]
process_climate=[]

for m in PROCESS_MATERIALS:
    er=elig.set_index('material').loc[m]
    for branch in PROCESS_BRANCHES:
        # require actual branch
        try:
            _=branch_bounds(er,branch)
        except Exception:
            continue
        for tau in TAU_HOURS:
            d=ldf_scores_for_problem(m,branch,BASE_MODES[PROCESS_MODE],tau)
            if len(d)<5:
                continue
            d['material']=m;d['branch']=branch;d['tau_h']=tau
            process_climate.append(d)

            rho=spearmanr(d.H_eq,d.H_LDF).statistic
            tau_rank=kendalltau(d.H_eq,d.H_LDF,variant='b').statistic
            rho_bal=spearmanr(d.H_eq,d.H_LDF_bal).statistic
            tau_bal=kendalltau(d.H_eq,d.H_LDF_bal,variant='b').statistic

            process_rows.append(dict(
                material=m,branch=branch,tau_h=tau,n_climates=len(d),
                spearman_H_vs_LDF=rho,
                kendall_H_vs_LDF=tau_rank,
                spearman_H_vs_LDF_bal=rho_bal,
                kendall_H_vs_LDF_bal=tau_bal,
            ))

process_summary=pd.DataFrame(process_rows)
process_detail=pd.concat(process_climate,ignore_index=True) if process_climate else pd.DataFrame()

process_summary.to_csv(OUT/'H_vs_LDF_rank_agreement.csv',index=False)
process_detail.to_csv(OUT/'H_vs_LDF_climate_scores.csv',index=False)

display(process_summary)

if len(process_summary):
    fig,ax=plt.subplots(figsize=(7,4.5))
    for (m,b),g in process_summary.groupby(['material','branch']):
        ax.plot(g.tau_h,g.kendall_H_vs_LDF,marker='o',label=f'{m} | {b}')
    ax.set_xscale('log')
    ax.set_ylim(-1,1)
    ax.axhline(0,linewidth=1)
    ax.set_xlabel('LDF time constant τ (h)')
    ax.set_ylabel('Kendall τ: H vs dynamic turnover')
    ax.set_title('Does equilibrium H preserve climate ordering under finite kinetics?')
    ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(FIG/'H_vs_LDF_rank_agreement.png',dpi=220)
    plt.show()



### Interpretation rule for Test D

- If rank agreement remains high across plausible \(\tau_k\), \(H\) is supported as an **equilibrium-opportunity ranking surrogate** under finite kinetics.
- If agreement collapses as \(\tau_k\) increases, the manuscript should state that the equilibrium-information requirement is **metric/process-timescale dependent**.
- Either outcome is informative. Do not call \(H_{\rm LDF}\) productivity.


## 6. Submission-facing summary table

In [ ]:

summary_rows=[]

# A
if 'mg_interp_summary' in globals() and len(mg_interp_summary):
    summary_rows.append(dict(
        test='A. Mg interpolation robustness',
        key_result=(
            f"{int(mg_interp_summary.n_C_negative.sum())}/"
            f"{int(mg_interp_summary.n_phases.sum())} scheme-phase results have C<0"
        ),
        status=(
            'PASS' if (mg_interp_summary.frac_C_negative==1).all()
            else 'SENSITIVE'
        )
    ))

# B
if 'C_pairs' in globals() and len(C_pairs):
    a=C_pairs[(np.abs(C_pairs.delta_Hsingle)>1e-12)&(np.abs(C_pairs.delta_Hfull)>1e-12)]
    exact=float((a.reversal==a.criterion_predicts_reversal).mean()) if len(a) else np.nan
    summary_rows.append(dict(
        test='B. C generalization',
        key_result=f'{len(Ccells)} class-C cells; exact decomposition criterion audit={exact:.3f}',
        status='DESCRIPTIVE'
    ))

# C
if 'cg' in globals():
    summary_rows.append(dict(
        test='C. Family-level inference',
        key_result=f"{cg.family.nunique()} independent class-C chemistry groups",
        status='LIMITED N'
    ))

# D
if 'process_summary' in globals() and len(process_summary):
    summary_rows.append(dict(
        test='D. H-process-proximal link',
        key_result=(
            'Kendall tau range = '
            f"{process_summary.kendall_H_vs_LDF.min():.3f} to "
            f"{process_summary.kendall_H_vs_LDF.max():.3f}"
        ),
        status='CHECK RANGE'
    ))

submission_summary=pd.DataFrame(summary_rows)
submission_summary.to_csv(OUT/'submission_gate_summary.csv',index=False)
display(submission_summary)

print('\nSaved outputs to:',OUT)
print('Saved figures to:',FIG)



## 7. Manuscript decision rules

### Mg mechanism
Use a strong wording only if \(C\) retains its sign across the endpoint-preserving interpolation stress tests.

### Generalization
Do not claim a universal mechanism merely because the algebraic reversal identity holds. The useful empirical question is whether the distribution and localization of \(C\) recur across independent class-C chemistries.

### Family inference
With only IRA, PEI-family, and MgAl in class C, family-level uncertainty remains intrinsically broad.

### \(H\) justification
The LDF analysis is a bridge from an equilibrium opportunity metric toward process behavior, not a replacement for a validated process model. A future direct comparison against published productivity from Sendi/Glaser-type process models would be stronger.
